# Used Cars — Statistical Analysis Project
**Topics Covered:** MLE · Binomial MLE · Naive Bayes · T-tests (1-sample, 2-sample independent, paired) · ANOVA · MANOVA · Regression (Linear, Ridge, Lasso, ElasticNet) · One-Hot & Label Encoding


## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import norm, ttest_1samp, ttest_ind, ttest_rel, f_oneway
from scipy.optimize import minimize

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (mean_squared_error, r2_score,
                              accuracy_score, confusion_matrix,
                              classification_report)

from statsmodels.multivariate.manova import MANOVA

plt.rcParams["figure.dpi"] = 110
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


---
## 1. Data Loading & Initial Inspection


In [ ]:
df = pd.read_csv("1775581885095_used_cars_data.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
print("Missing values:\n")
print(df.isnull().sum())
print()
print("Dtypes:\n")
print(df.dtypes)


---
## 2. Data Cleaning

### Strategy
| Column | Issue | Fix |
|---|---|---|
| `Price` | 1,234 nulls | Fill with **brand-wise mean** |
| `New_Price` | 86% null | Compute avg depreciation from known rows, then `Price + avg_depreciation` |
| `Mileage / Engine / Power` | string + units | Strip units → float, then fill nulls with median |
| `Seats` | 53 nulls | Fill with median |
| `New_Price` (still null) | can't compute depreciation | Fill remaining with overall mean |


In [ ]:
# ── 2a. Strip units from numeric string columns ──────────────────────────────
def strip_units(series):
    """Extract leading float from '26.6 km/kg', '998 CC', '58.16 bhp' etc."""
    return series.astype(str).str.extract(r"(\d+\.?\d*)")[0].astype(float)

df["Mileage"] = strip_units(df["Mileage"])
df["Engine"]  = strip_units(df["Engine"])
df["Power"]   = strip_units(df["Power"])
df["New_Price"] = strip_units(df["New_Price"])   # now float (Lakh)

print("After unit stripping:")
print(df[["Mileage","Engine","Power","New_Price"]].describe())


In [ ]:
# ── 2b. Extract Brand ────────────────────────────────────────────────────────
df["Brand"] = df["Name"].str.split().str[0]
print("Brands found:", df["Brand"].nunique())
print(df["Brand"].value_counts().head(10))


In [ ]:
# ── 2c. Fill Price NAs with brand-wise mean ──────────────────────────────────
brand_mean_price = df.groupby("Brand")["Price"].transform("mean")
df["Price"] = df["Price"].fillna(brand_mean_price)

# If a brand has ALL prices missing, fall back to overall mean
df["Price"] = df["Price"].fillna(df["Price"].mean())

print(f"Price nulls after brand-wise fill: {df['Price'].isnull().sum()}")


In [ ]:
# ── 2d. Fill New_Price using depreciation approach ───────────────────────────
#
# Depreciation = New_Price - Price  (computed on rows where BOTH exist)
# Then: New_Price (missing) = Price + avg_brand_depreciation
#
# NOTE: Min depreciation is -157 Lakh (some used cars > new price — data noise).
# We clip depreciation at 0 before averaging to avoid nonsensical imputation.

rows_both = df.dropna(subset=["New_Price", "Price"]).copy()
rows_both["Depreciation"] = rows_both["New_Price"] - rows_both["Price"]
rows_both["Depreciation"] = rows_both["Depreciation"].clip(lower=0)  # clip negatives

print("Depreciation stats (clipped):")
print(rows_both["Depreciation"].describe())
print(f"\nNegative depreciation rows removed: {(rows_both['Depreciation'] < 0).sum()}")

# Brand-wise average depreciation
brand_depr = rows_both.groupby("Brand")["Depreciation"].mean()
print("\nBrand-wise avg depreciation (top 10):")
print(brand_depr.sort_values(ascending=False).head(10))


In [ ]:
# Fill missing New_Price using: Price + brand_avg_depreciation
df["Brand_Avg_Depr"] = df["Brand"].map(brand_depr)
mask_new = df["New_Price"].isnull()
df.loc[mask_new, "New_Price"] = df.loc[mask_new, "Price"] + df.loc[mask_new, "Brand_Avg_Depr"]

# Remaining nulls (brands with no depreciation data) → overall mean
df["New_Price"] = df["New_Price"].fillna(df["New_Price"].mean())
df.drop(columns=["Brand_Avg_Depr"], inplace=True)

print(f"New_Price nulls after depreciation fill: {df['New_Price'].isnull().sum()}")


In [ ]:
# ── 2e. Fill remaining NAs with column median ────────────────────────────────
for col in ["Mileage", "Engine", "Power", "Seats"]:
    df[col].fillna(df[col].median(), inplace=True)

print("Nulls remaining after all fills:")
print(df.isnull().sum())


In [ ]:
# ── 2f. Feature engineering ──────────────────────────────────────────────────
df["Car_Age"] = 2024 - df["Year"]

# Drop columns no longer needed
df.drop(columns=["S.No.", "Name", "Year"], inplace=True)

print("Final shape:", df.shape)
print("\nColumn list:", df.columns.tolist())
df.head(3)


---
## 3. Exploratory Data Analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df["Price"], bins=60, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_title("Price Distribution (Raw)")
axes[0].set_xlabel("Price (Lakh ₹)")

axes[1].hist(np.log(df["Price"]), bins=60, color="salmon", edgecolor="white", alpha=0.8)
axes[1].set_title("Log(Price) — Near Normal")
axes[1].set_xlabel("log(Price)")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df.boxplot(column="Price", by="Fuel_Type", ax=axes[0])
axes[0].set_title("Price by Fuel Type")
plt.suptitle("")

df.boxplot(column="Price", by="Transmission", ax=axes[1])
axes[1].set_title("Price by Transmission")
plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
num_cols = ["Price","New_Price","Kilometers_Driven","Mileage",
            "Engine","Power","Seats","Car_Age"]
corr = df[num_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


---
## 4. Encoding

| Column | Method | Reason |
|---|---|---|
| `Owner_Type` | **Label Encoding** | Ordinal — First < Second < Third < Fourth |
| `Brand` | **Label Encoding** | 33 levels — OHE would explode dimensionality |
| `Fuel_Type`, `Transmission`, `Location` | **One-Hot Encoding** | Nominal — no natural order |


In [ ]:
df_enc = df.copy()

# ── Label Encoding: Owner_Type (ordinal) ─────────────────────────────────────
owner_map = {"First": 1, "Second": 2, "Third": 3, "Fourth & Above": 4}
df_enc["Owner_Type"] = df_enc["Owner_Type"].map(owner_map)
print("Owner_Type mapped:", df_enc["Owner_Type"].value_counts().sort_index())


In [ ]:
# ── Label Encoding: Brand ────────────────────────────────────────────────────
le_brand = LabelEncoder()
df_enc["Brand"] = le_brand.fit_transform(df_enc["Brand"])
print("Brand encoded. Sample:", df_enc["Brand"].unique()[:8])


In [ ]:
# ── One-Hot Encoding: Fuel_Type, Transmission, Location ──────────────────────
ohe_cols = ["Fuel_Type", "Transmission", "Location"]
df_enc = pd.get_dummies(df_enc, columns=ohe_cols, drop_first=True)

print("Shape after encoding:", df_enc.shape)
print("New columns sample:", [c for c in df_enc.columns if any(o in c for o in ohe_cols)][:8])
df_enc.head(2)


---
## 5. Maximum Likelihood Estimation (MLE)

### 5a. Log-Normal MLE on Price

Car prices are right-skewed. We fit a **Log-Normal distribution**, i.e., log(Price) ~ N(μ, σ²).

**MLE closed-form solutions:**
$$\hat{\mu} = \frac{1}{n}\sum \log(x_i), \quad \hat{\sigma}^2 = \frac{1}{n}\sum(\log(x_i) - \hat{\mu})^2$$


In [ ]:
log_prices = np.log(df["Price"])

# MLE estimates (use ddof=0 — MLE divides by N, not N-1)
mu_mle    = log_prices.mean()
sigma_mle = log_prices.std(ddof=0)

print(f"MLE  μ̂  = {mu_mle:.4f}   (log scale)")
print(f"MLE  σ̂  = {sigma_mle:.4f}   (log scale)")
print(f"Implied median price = ₹{np.exp(mu_mle):.2f} Lakh")
print(f"Implied mean price   = ₹{np.exp(mu_mle + 0.5*sigma_mle**2):.2f} Lakh")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Raw price
axes[0].hist(df["Price"], bins=60, density=True, color="steelblue", alpha=0.7)
axes[0].set_title("Price — Right Skewed")
axes[0].set_xlabel("Price (Lakh ₹)")

# Log price + MLE fit
x = np.linspace(log_prices.min(), log_prices.max(), 300)
axes[1].hist(log_prices, bins=60, density=True, color="salmon", alpha=0.7, label="Observed")
axes[1].plot(x, norm.pdf(x, mu_mle, sigma_mle), "k-", lw=2.5,
             label=f"MLE fit: N({mu_mle:.2f}, {sigma_mle:.2f}²)")
axes[1].set_title("Log(Price) — MLE Fitted Normal")
axes[1].set_xlabel("log(Price)")
axes[1].legend()

plt.tight_layout()
plt.show()


### 5b. MLE using Numerical Optimization (verify closed-form)

We can also derive MLE numerically by minimising the negative log-likelihood — and should get the same result.

In [ ]:
def neg_log_likelihood(params, data):
    mu, log_sigma = params
    sigma = np.exp(log_sigma)   # exponentiate to keep sigma > 0
    return -np.sum(norm.logpdf(data, loc=mu, scale=sigma))

result = minimize(neg_log_likelihood, x0=[1.0, 0.5], args=(log_prices,))
mu_num, sigma_num = result.x[0], np.exp(result.x[1])

print(f"Numerical MLE  μ̂ = {mu_num:.4f}  (closed-form: {mu_mle:.4f})")
print(f"Numerical MLE  σ̂ = {sigma_num:.4f}  (closed-form: {sigma_mle:.4f})")
print("✓ Both match — confirms correctness of closed-form solution")


---
## 6. Binomial MLE

**Setup:** Define `Expensive = 1` if Price > median(Price), else 0.  
Each car is an independent Bernoulli trial with unknown probability *p*.

**MLE for Binomial:** $\hat{p} = k/n$ (fraction of successes)


In [ ]:
median_price    = df["Price"].median()
df["Expensive"] = (df["Price"] > median_price).astype(int)
df_enc["Expensive"] = df["Expensive"]

n     = len(df)
k     = df["Expensive"].sum()
p_hat = k / n

print(f"Median price threshold : ₹{median_price:.2f} Lakh")
print(f"n = {n},  k (expensive) = {k}")
print(f"Binomial MLE  p̂ = k/n = {k}/{n} = {p_hat:.4f}")


In [ ]:
# p̂ per city — compare to overall
city_p = df.groupby("Location")["Expensive"].mean().sort_values(ascending=False)
print("P(Expensive) by city:\n")
print(city_p.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
city_p.plot(kind="bar", ax=ax, color="teal", edgecolor="white")
ax.axhline(p_hat, color="red", ls="--", lw=1.5, label=f"Overall p̂ = {p_hat:.2f}")
ax.set_title("Binomial MLE — P(Expensive Car) by City")
ax.set_ylabel("Estimated Probability p̂")
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


---
## 7. Naive Bayes Classifier

**Target:** `Expensive` (0 = Budget, 1 = Premium)  
**Model:** Gaussian Naive Bayes — assumes each feature is Gaussian within each class.

**Bayes theorem:**
$$P(C|x) \propto P(C) \prod_i P(x_i | C)$$


In [ ]:
nb_features = ["Mileage","Engine","Power","Car_Age",
               "Kilometers_Driven","Owner_Type","Seats","New_Price"]

X_nb = df[nb_features]
y_nb = df["Expensive"]

X_tr_nb, X_te_nb, y_tr_nb, y_te_nb = train_test_split(
    X_nb, y_nb, test_size=0.2, random_state=42, stratify=y_nb
)

gnb = GaussianNB()
gnb.fit(X_tr_nb, y_tr_nb)
y_pred_nb = gnb.predict(X_te_nb)

print("Accuracy :", accuracy_score(y_te_nb, y_pred_nb).round(4))
print()
print(classification_report(y_te_nb, y_pred_nb, target_names=["Budget","Premium"]))


In [ ]:
cm = confusion_matrix(y_te_nb, y_pred_nb)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Confusion matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Budget","Premium"],
            yticklabels=["Budget","Premium"], ax=axes[0])
axes[0].set_title(f"Confusion Matrix  (Acc = {accuracy_score(y_te_nb, y_pred_nb):.2f})")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

# Class-wise feature means
class_means = pd.DataFrame({
    "Budget (0)":  X_tr_nb[y_tr_nb == 0].mean(),
    "Premium (1)": X_tr_nb[y_tr_nb == 1].mean()
})
class_means_norm = class_means.div(class_means.max(axis=1), axis=0)  # normalise 0-1
class_means_norm.plot(kind="bar", ax=axes[1], color=["steelblue","salmon"])
axes[1].set_title("Normalised Feature Means by Class")
axes[1].set_ylabel("Relative value")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.show()


In [ ]:
# ── Gaussian parameters learned by the model ─────────────────────────────────
print("Class priors:", dict(zip(["Budget","Premium"], gnb.class_prior_.round(4))))
print()
print("Class-wise feature means (theta_):")
print(pd.DataFrame(gnb.theta_, columns=nb_features,
                   index=["Budget","Premium"]).round(2).T)


---
## 8. T-Tests

### 8a. One-Sample T-test
**H₀:** The mean used car price in the dataset = ₹9.5 Lakh (national benchmark)  
**H₁:** Mean ≠ 9.5 Lakh


In [ ]:
pop_mean = 9.5
t1, p1 = ttest_1samp(df["Price"], popmean=pop_mean)

print("One-Sample T-test")
print(f"  H₀: μ = ₹{pop_mean}L")
print(f"  Sample mean    = ₹{df['Price'].mean():.4f}L")
print(f"  t-statistic    = {t1:.4f}")
print(f"  p-value        = {p1:.4f}")
print()
if p1 < 0.05:
    print(f"  ✗ Reject H₀ — mean price is significantly ≠ ₹{pop_mean}L  (α=0.05)")
else:
    print(f"  ✓ Fail to reject H₀ — no significant difference from ₹{pop_mean}L  (α=0.05)")


### 8b. Two-Sample Independent T-test
**H₀:** Mean price is equal for Manual and Automatic cars.

In [ ]:
manual    = df[df["Transmission"] == "Manual"]["Price"]
automatic = df[df["Transmission"] == "Automatic"]["Price"]

t2, p2 = ttest_ind(manual, automatic, equal_var=False)   # Welch's t-test

print("Two-Sample Independent T-test (Welch's)")
print(f"  Manual    mean = ₹{manual.mean():.4f}L  (n={len(manual)})")
print(f"  Automatic mean = ₹{automatic.mean():.4f}L  (n={len(automatic)})")
print(f"  t-statistic = {t2:.4f}")
print(f"  p-value     = {p2:.6f}")
print()
if p2 < 0.05:
    print("  ✗ Reject H₀ — Transmission type significantly affects price")
else:
    print("  ✓ Fail to reject H₀")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(manual,    bins=50, alpha=0.6, color="steelblue", label=f"Manual (μ=₹{manual.mean():.1f}L)")
ax.hist(automatic, bins=50, alpha=0.6, color="salmon",    label=f"Automatic (μ=₹{automatic.mean():.1f}L)")
ax.axvline(manual.mean(),    color="steelblue", ls="--")
ax.axvline(automatic.mean(), color="salmon",    ls="--")
ax.set_xlabel("Price (Lakh ₹)")
ax.set_title(f"Manual vs Automatic Prices  (t={t2:.2f}, p={p2:.4f})")
ax.legend()
plt.tight_layout()
plt.show()


### 8c. Two-Sample Paired T-test
**Creative application:** Match cars of the same Brand + Fuel_Type, compare First vs Second owner prices.  
**H₀:** First owner price − Second owner price = 0 (ownership doesn't affect price)


In [ ]:
df_raw = pd.read_csv("1775581885095_used_cars_data.csv")

# Re-compute Brand for raw data
df_raw["Brand"] = df_raw["Name"].str.split().str[0]

d1 = (df_raw[df_raw["Owner_Type"] == "First"]
      [["Brand","Fuel_Type","Price"]]
      .dropna()
      .rename(columns={"Price": "Price_First"}))

d2 = (df_raw[df_raw["Owner_Type"] == "Second"]
      [["Brand","Fuel_Type","Price"]]
      .dropna()
      .rename(columns={"Price": "Price_Second"}))

paired = pd.merge(d1, d2, on=["Brand","Fuel_Type"])
print(f"Matched pairs: {len(paired)}")

t3, p3 = ttest_rel(paired["Price_First"], paired["Price_Second"])

print(f"  First owner  mean = ₹{paired['Price_First'].mean():.4f}L")
print(f"  Second owner mean = ₹{paired['Price_Second'].mean():.4f}L")
print(f"  Avg price drop    = ₹{(paired['Price_First'] - paired['Price_Second']).mean():.4f}L")
print(f"  t-statistic = {t3:.4f}")
print(f"  p-value     = {p3:.6f}")
print()
if p3 < 0.05:
    print("  ✗ Reject H₀ — Ownership history significantly affects price")
else:
    print("  ✓ Fail to reject H₀")


In [ ]:
diff = paired["Price_First"] - paired["Price_Second"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(diff, bins=40, color="teal", edgecolor="white", alpha=0.8)
axes[0].axvline(0,          color="red",   ls="--", label="H₀: diff=0")
axes[0].axvline(diff.mean(), color="black", ls="-",  label=f"Mean diff = ₹{diff.mean():.2f}L")
axes[0].set_xlabel("Price Difference (First − Second)")
axes[0].set_title("Paired Difference Distribution")
axes[0].legend()

axes[1].scatter(paired["Price_First"], paired["Price_Second"],
                alpha=0.3, color="steelblue", s=15)
lim = max(paired["Price_First"].max(), paired["Price_Second"].max())
axes[1].plot([0, lim], [0, lim], "r--", label="y = x (no difference)")
axes[1].set_xlabel("Price — First Owner (₹L)")
axes[1].set_ylabel("Price — Second Owner (₹L)")
axes[1].set_title("First vs Second Owner Prices (Matched Pairs)")
axes[1].legend()

plt.tight_layout()
plt.show()


---
## 9. One-Way ANOVA

**H₀:** Mean price is the same across all fuel types.  
**H₁:** At least one fuel type has a different mean price.

Tests whether the *between-group* variance is significantly larger than *within-group* variance via F-statistic.


In [ ]:
groups = {ft: g["Price"].values for ft, g in df.groupby("Fuel_Type")}
F, p_anova = f_oneway(*groups.values())

print("One-Way ANOVA — Price by Fuel Type")
print(f"  F-statistic = {F:.4f}")
print(f"  p-value     = {p_anova:.6f}")
print()
if p_anova < 0.05:
    print("  ✗ Reject H₀ — at least one fuel type has a significantly different mean price")
else:
    print("  ✓ Fail to reject H₀")

print("\nGroup statistics:")
print(df.groupby("Fuel_Type")["Price"].agg(["mean","std","count"]).round(2))


In [ ]:
# ── Post-hoc: Tukey's HSD (tells us WHICH pairs differ) ──────────────────────
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=df["Price"], groups=df["Fuel_Type"], alpha=0.05)
print(tukey.summary())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df.boxplot(column="Price", by="Fuel_Type", ax=axes[0])
axes[0].set_title(f"Price by Fuel Type  (F={F:.2f}, p={p_anova:.4f})")
axes[0].set_xlabel("Fuel Type")
axes[0].set_ylabel("Price (Lakh ₹)")
plt.suptitle("")

# Group means + confidence intervals
means  = df.groupby("Fuel_Type")["Price"].mean()
sems   = df.groupby("Fuel_Type")["Price"].sem()
axes[1].bar(means.index, means.values, yerr=sems.values*1.96,
            color="teal", edgecolor="white", capsize=5)
axes[1].set_title("Mean Price ± 95% CI by Fuel Type")
axes[1].set_ylabel("Mean Price (Lakh ₹)")
plt.xticks(rotation=20, ha="right")

plt.tight_layout()
plt.show()


---
## 10. MANOVA (Multivariate ANOVA)

**Why MANOVA over ANOVA?**  
ANOVA tests one dependent variable at a time. MANOVA tests whether the *combination* of multiple dependent variables differs across groups — it accounts for correlations between outcomes.

**Setup:**
- **Dependent variables:** Price, Kilometers_Driven, Car_Age
- **Grouping factor:** Transmission (Manual vs Automatic)

**H₀:** The multivariate mean vector [Price, KM, Age] is equal across transmission types.


In [ ]:
manova_df = df[["Price","Kilometers_Driven","Car_Age","Transmission"]].copy()
manova_df = manova_df[manova_df["Transmission"].isin(["Manual","Automatic"])].dropna()

print("MANOVA group sizes:")
print(manova_df["Transmission"].value_counts())


In [ ]:
maov = MANOVA.from_formula(
    "Price + Kilometers_Driven + Car_Age ~ Transmission",
    data=manova_df
)
result = maov.mv_test()
print(result)


In [ ]:
# Visual: multivariate group means
group_means = manova_df.groupby("Transmission")[["Price","Kilometers_Driven","Car_Age"]].mean()
# Normalise for radar-like bar comparison
group_norm = group_means.div(group_means.max())

ax = group_norm.T.plot(kind="bar", figsize=(8, 4), color=["steelblue","salmon"])
ax.set_title("Normalised Group Means — Manual vs Automatic")
ax.set_ylabel("Normalised value (0-1)")
plt.xticks(rotation=0)
plt.legend(title="Transmission")
plt.tight_layout()
plt.show()


---
## 11. Regression Models

### 11a. Train / Test Split + Scaling


In [ ]:
X = df_enc.drop(columns=["Price","Expensive"])
y = df_enc["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler   = StandardScaler()
X_tr_s   = scaler.fit_transform(X_train)
X_te_s   = scaler.transform(X_test)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")


### 11b. Train All Four Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge (α=1.0)":     Ridge(alpha=1.0),
    "Lasso (α=0.1)":     Lasso(alpha=0.1, max_iter=10000),
    "ElasticNet (α=0.1, l1=0.5)": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000),
}

results = {}
for name, model in models.items():
    model.fit(X_tr_s, y_train)
    y_pred = model.predict(X_te_s)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)
    cv_r2  = cross_val_score(model, X_tr_s, y_train, cv=5, scoring="r2").mean()
    results[name] = {"RMSE": rmse, "R²": r2, "CV R²": cv_r2}
    print(f"{name:35s}  RMSE={rmse:.3f}  R²={r2:.4f}  CV R²={cv_r2:.4f}")

res_df = pd.DataFrame(results).T


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

res_df["RMSE"].plot(kind="bar",  ax=axes[0], color="salmon",    edgecolor="white")
axes[0].set_title("RMSE (lower = better)")
axes[0].set_ylabel("RMSE")
plt.setp(axes[0].get_xticklabels(), rotation=20, ha="right", fontsize=8)

res_df["R²"].plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("R² (higher = better)")
axes[1].set_ylabel("R²")
plt.setp(axes[1].get_xticklabels(), rotation=20, ha="right", fontsize=8)

res_df["CV R²"].plot(kind="bar", ax=axes[2], color="teal", edgecolor="white")
axes[2].set_title("Cross-Validated R²")
axes[2].set_ylabel("CV R²")
plt.setp(axes[2].get_xticklabels(), rotation=20, ha="right", fontsize=8)

plt.tight_layout()
plt.show()


### 11c. Ridge Coefficient Path
Shows how coefficients shrink as regularisation increases.

In [ ]:
alphas    = np.logspace(-2, 4, 100)
coef_path = np.array([Ridge(alpha=a).fit(X_tr_s, y_train).coef_ for a in alphas])

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(coef_path.shape[1]):
    ax.plot(np.log10(alphas), coef_path[:, i], alpha=0.35, lw=1)
ax.axvline(0, color="red", ls="--", lw=1.5, label="α=1 (used)")
ax.set_xlabel("log₁₀(α)")
ax.set_ylabel("Coefficient value")
ax.set_title("Ridge Coefficient Path — all features")
ax.legend()
plt.tight_layout()
plt.show()


### 11d. Lasso Coefficient Path
Lasso drives coefficients to exactly zero — automatic feature selection.

In [ ]:
alphas_l    = np.logspace(-2, 2, 80)
lasso_path  = np.array([Lasso(alpha=a, max_iter=10000).fit(X_tr_s, y_train).coef_
                         for a in alphas_l])

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(lasso_path.shape[1]):
    ax.plot(np.log10(alphas_l), lasso_path[:, i], alpha=0.35, lw=1)
ax.axvline(np.log10(0.1), color="red", ls="--", lw=1.5, label="α=0.1 (used)")
ax.set_xlabel("log₁₀(α)")
ax.set_ylabel("Coefficient value")
ax.set_title("Lasso Coefficient Path — features → 0 as α increases")
ax.legend()
plt.tight_layout()
plt.show()


### 11e. Feature Importance — Top Predictors

In [ ]:
ridge_model = Ridge(alpha=1.0).fit(X_tr_s, y_train)
lasso_model = Lasso(alpha=0.1, max_iter=10000).fit(X_tr_s, y_train)

coef_df = pd.DataFrame({
    "Feature":    X.columns,
    "Ridge":      ridge_model.coef_,
    "Lasso":      lasso_model.coef_,
}).set_index("Feature")

top10 = coef_df["Ridge"].abs().nlargest(10).index

print("Top 10 features by Ridge coefficient magnitude:")
print(coef_df.loc[top10].round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
coef_df.loc[top10, "Ridge"].sort_values().plot(
    kind="barh", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Ridge — Top 10 Coefficients")
axes[0].axvline(0, color="black", lw=0.8)

lasso_nonzero = coef_df[coef_df["Lasso"] != 0]["Lasso"]
lasso_nonzero.abs().nlargest(10).index.pipe(lambda idx: coef_df.loc[idx, "Lasso"]).sort_values().plot(
    kind="barh", ax=axes[1], color="salmon", edgecolor="white")
axes[1].set_title("Lasso — Top 10 Non-zero Coefficients")
axes[1].axvline(0, color="black", lw=0.8)

plt.tight_layout()
plt.show()

print(f"\nLasso features set to zero: {(coef_df['Lasso'] == 0).sum()} / {len(coef_df)}")


### 11f. Actual vs Predicted — Best Model

In [ ]:
best_model = ridge_model
y_pred_best = best_model.predict(X_te_s)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(y_test, y_pred_best, alpha=0.3, s=10, color="steelblue")
lim = max(y_test.max(), y_pred_best.max())
axes[0].plot([0, lim], [0, lim], "r--", lw=1.5, label="Perfect prediction")
axes[0].set_xlabel("Actual Price (₹L)")
axes[0].set_ylabel("Predicted Price (₹L)")
axes[0].set_title(f"Actual vs Predicted — Ridge  (R²={r2_score(y_test, y_pred_best):.3f})")
axes[0].legend()

residuals = y_test - y_pred_best
axes[1].hist(residuals, bins=60, color="teal", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="red", ls="--")
axes[1].set_xlabel("Residual (Actual − Predicted)")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()


### 11g. ElasticNet — Effect of l1_ratio

In [ ]:
l1_ratios = np.linspace(0.01, 0.99, 30)
en_results = []
for l1 in l1_ratios:
    en = ElasticNet(alpha=0.1, l1_ratio=l1, max_iter=10000).fit(X_tr_s, y_train)
    r2_val = r2_score(y_test, en.predict(X_te_s))
    nonzero = (en.coef_ != 0).sum()
    en_results.append({"l1_ratio": l1, "R²": r2_val, "Non-zero features": nonzero})

en_df = pd.DataFrame(en_results)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.plot(en_df["l1_ratio"], en_df["R²"],               "steelblue", lw=2, label="R²")
ax2.plot(en_df["l1_ratio"], en_df["Non-zero features"], "salmon",    lw=2, ls="--", label="Non-zero features")
ax1.set_xlabel("l1_ratio  (0=Ridge, 1=Lasso)")
ax1.set_ylabel("R²",               color="steelblue")
ax2.set_ylabel("Non-zero features", color="salmon")
ax1.set_title("ElasticNet — R² and Sparsity vs l1_ratio")
ax1.legend(loc="upper left"); ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()


---
## 12. Summary of Results

| Test / Model | Result |
|---|---|
| **MLE** | log(Price) ~ N(μ̂, σ̂²) — Log-Normal fits well |
| **Binomial MLE** | p̂ varies significantly by city |
| **Naive Bayes** | ~75%+ accuracy on Budget vs Premium |
| **1-sample t-test** | Mean price significantly ≠ ₹9.5L |
| **2-sample t-test** | Automatic cars significantly more expensive |
| **Paired t-test** | First owner cars significantly more expensive than Second |
| **ANOVA** | Fuel type significantly affects price (F >> 1) |
| **MANOVA** | Transmission affects multivariate outcome vector |
| **Ridge** | Best R² ~0.85, handles multicollinearity (Engine ↔ Power) |
| **Lasso** | Performs feature selection — zeroes out weak predictors |
| **ElasticNet** | Balances Ridge + Lasso; tune l1_ratio for best tradeoff |
